# Notebook 09 — Monitoring and Structural Breaks

**References:** W&H §11.4–11.5

**New engine function:** `engine.interventions.kalman_filter_monitor`

The sequential Bayes factor monitoring statistic H_t detects when the data
is better explained by a reference model with inflated observation variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from engine.interventions import kalman_filter_monitor
from engine.models import make_local_level
from engine.simulate import simulate

## 1. Sequential Bayes Factor

At each time step $t$, compare the one-step predictive density:

$$H_t = \frac{p(y_t \mid y_{1:t-1}, M_1)}{p(y_t \mid y_{1:t-1}, M_0)}$$

- $M_1$: current model with observation variance $V$
- $M_0$: reference "robustness" model with inflated variance $\omega V$, $\omega \gg 1$

When $H_t < 1$, the data is better explained by $M_0$ at step $t$.

The running product:
$$L_t = \prod_{s=1}^{t} H_s$$

falls monotonically when the data repeatedly favours $M_0$.
An alert is triggered when $L_t < \text{threshold}$.

**Connection to CUSUM:** $-\log L_t$ is a Bayesian analogue of the CUSUM statistic.
The threshold $L < 0.1$ corresponds to a Bayes factor $> 10$ in favour of model inadequacy.

**Reference:** W&H §11.4, eq. (11.22)

In [ ]:
spec = make_local_level(V=1.0, W_level=0.1)
sim_stable = simulate(spec, n=200, seed=0)
mr_stable = kalman_filter_monitor(spec, sim_stable.y, inflation=100.0, threshold=0.1)
print(f"Alerts on stable series: {mr_stable.alert.sum()} / 200")

In [ ]:
rng = np.random.default_rng(42)
T = 300
y_break = np.zeros((T, 1))
level = 0.0
for t in range(T):
    w_var = 0.1 if t < 200 else 10.0
    level += rng.normal(scale=np.sqrt(w_var))
    y_break[t, 0] = level + rng.normal(scale=1.0)

mr_break = kalman_filter_monitor(spec, y_break, inflation=100.0, threshold=0.1)
alert_times = np.where(mr_break.alert)[0]
print(f"First alert at t={alert_times[0] if len(alert_times) > 0 else 'never'}")

In [ ]:
t_arr = np.arange(T)
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
axes[0].plot(t_arr, y_break[:, 0], lw=0.5, label="obs")
axes[0].axvline(200, color="red", ls="--", lw=0.8, label="structural break (t=200)")
axes[0].legend()
axes[0].set_title("Observations")

axes[1].semilogy(t_arr, mr_break.H, lw=0.8, label="H_t (per-step Bayes factor)")
axes[1].axhline(1.0, lw=0.5, color="k", ls="--")
axes[1].axvline(200, color="red", ls="--", lw=0.8)
axes[1].legend()
axes[1].set_title("Per-step Bayes factor H_t")

axes[2].semilogy(t_arr, np.clip(mr_break.L, 1e-300, None), lw=1.0, label="L_t (cumulative product)")
axes[2].axhline(0.1, color="orange", ls="--", lw=0.8, label="threshold=0.1")
axes[2].axvline(200, color="red", ls="--", lw=0.8)
axes[2].legend()
axes[2].set_title("Cumulative product L_t and alert times")
plt.tight_layout()
plt.show()

## Exercises

**Exercise 1** — Threshold sensitivity:
Run `kalman_filter_monitor` on the structural-break series with
`threshold` values 0.01, 0.1, 0.5. For each, record the first alert time.

```python
# YOUR CODE HERE
thresholds = [0.01, 0.1, 0.5]
first_alerts = []
for thr in thresholds:
    mr = kalman_filter_monitor(spec, y_break, inflation=100.0, threshold=thr)
    alerts = np.where(mr.alert)[0]
    first_alerts.append(alerts[0] if len(alerts) > 0 else T)
# Plot first_alerts vs thresholds
```

**Exercise 2** — Inflation sensitivity:
Fix `threshold=0.1`. Run the monitor with `inflation` values 10, 100, 1000.
How does the first-alert time change?

**Exercise 3 (challenge)** — Resetting monitor:
After an alert, practitioners often reset L_t = 1 and restart monitoring.
Implement a resetting version and show it detects a second structural break.

In [ ]:
# Exercise 1 scaffold
# YOUR CODE HERE
thresholds = [0.01, 0.1, 0.5]
first_alerts = []
for thr in thresholds:
    mr = kalman_filter_monitor(spec, y_break, inflation=100.0, threshold=thr)
    alerts = np.where(mr.alert)[0]
    first_alerts.append(int(alerts[0]) if len(alerts) > 0 else T)
print("threshold -> first alert:", list(zip(thresholds, first_alerts)))